[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safe-t-ai/safe-t-ai.github.io/blob/main/notebooks/01_fetch_data.ipynb)

# 01 — Fetch Raw Durham Data

Fetches and writes three raw data files:
- `backend/data/raw/durham_census_tracts.geojson` — Census ACS demographics + TIGER geometries
- `backend/data/raw/ncdot_nonmotorist_durham.csv` — NCDOT crash records (pedestrian/cyclist)
- `backend/data/raw/osm_infrastructure.json` — OSM infrastructure features, spatial-joined to tracts

Run cells top to bottom. Requires Colab secrets `CENSUS_API_KEY` and `GITHUB_TOKEN_SAFET`.

In [ ]:
# Install dependencies
%pip install -q geopandas requests shapely osmnx

In [ ]:
# Bootstrap repo — clones if absent, adds backend/ to sys.path
import subprocess, sys
from pathlib import Path

NOTEBOOKS_URL = "https://raw.githubusercontent.com/safe-t-ai/safe-t-ai.github.io/main/notebooks/colab_utils.py"
utils_path = Path("/content/colab_utils.py")
if not utils_path.exists():
    import urllib.request
    urllib.request.urlretrieve(NOTEBOOKS_URL, str(utils_path))

sys.path.insert(0, "/content")
import colab_utils

repo = colab_utils.prepare_notebook(pull_latest=True)
print(f"Repo ready at {repo}")

In [ ]:
# Read CENSUS_API_KEY from Colab secrets
from google.colab import userdata
import os

os.environ["CENSUS_API_KEY"] = userdata.get("CENSUS_API_KEY")
print("CENSUS_API_KEY loaded.")

## 1. Census ACS Demographics + TIGER Geometries

In [ ]:
import requests
import geopandas as gpd
import pandas as pd
from config import RAW_DATA_DIR, CENSUS_API_KEY, CENSUS_VINTAGE, TIGER_VINTAGE, TIGER_TRACTS_LAYER
from utils.freshness import write_meta

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CENSUS = RAW_DATA_DIR / 'durham_census_tracts.geojson'

state_fips = '37'
county_fips = '063'
base_url = f"https://api.census.gov/data/{CENSUS_VINTAGE}/acs/acs5"

variables = [
    'B01003_001E',  # Total population
    'B19013_001E',  # Median household income
    'B02001_002E',  # White alone
    'B02001_003E',  # Black/African American alone
    'B03003_003E',  # Hispanic/Latino
]

params = {
    'get': ','.join(['NAME'] + variables),
    'for': 'tract:*',
    'in': f'state:{state_fips} county:{county_fips}',
    'key': CENSUS_API_KEY,
}

print("Fetching census demographic data...")
response = requests.get(base_url, params=params)
response.raise_for_status()

data = response.json()
df = pd.DataFrame(data[1:], columns=data[0])
df = df.rename(columns={
    'B01003_001E': 'total_population',
    'B19013_001E': 'median_income',
    'B02001_002E': 'white_population',
    'B02001_003E': 'black_population',
    'B03003_003E': 'hispanic_population',
})

numeric_cols = ['total_population', 'median_income', 'white_population',
                'black_population', 'hispanic_population']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['pct_white'] = (df['white_population'] / df['total_population'] * 100).round(1)
df['pct_black'] = (df['black_population'] / df['total_population'] * 100).round(1)
df['pct_hispanic'] = (df['hispanic_population'] / df['total_population'] * 100).round(1)
df['pct_minority'] = (100 - df['pct_white']).round(1)
df['tract_id'] = df['state'] + df['county'] + df['tract']

print(f"  {len(df)} tracts with demographics")

In [ ]:
# Fetch TIGER geometries and merge
tiger_url = (
    f"https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb"
    f"/tigerWMS_ACS{TIGER_VINTAGE}/MapServer/{TIGER_TRACTS_LAYER}/query"
)
tiger_params = {
    'where': f"STATE='{state_fips}' AND COUNTY='{county_fips}'",
    'outFields': '*',
    'f': 'geojson',
    'returnGeometry': 'true',
}

print("Fetching tract geometries...")
geom_response = requests.get(tiger_url, params=tiger_params)
geom_response.raise_for_status()
geojson_data = geom_response.json()

if 'features' not in geojson_data:
    raise RuntimeError(
        f"TIGER service tigerWMS_ACS{TIGER_VINTAGE} returned no features. "
        f"Response: {str(geojson_data)[:500]}"
    )

gdf = gpd.GeoDataFrame.from_features(geojson_data['features'])
gdf['tract_id'] = gdf['STATE'] + gdf['COUNTY'] + gdf['TRACT']
gdf = gdf.merge(df, on='tract_id', how='left')

# Replace Census sentinel income values (-666666666 = missing)
sentinel_mask = gdf['median_income'] < 0
if sentinel_mask.any():
    county_median = gdf.loc[~sentinel_mask, 'median_income'].median()
    print(f"  Replacing {sentinel_mask.sum()} sentinel income values with county median ${county_median:,.0f}")
    gdf.loc[sentinel_mask, 'median_income'] = county_median

# Drop water-only / institutional tracts
zero_pop = gdf['total_population'] == 0
if zero_pop.any():
    print(f"  Dropping {zero_pop.sum()} tracts with 0 population")
    gdf = gdf[~zero_pop].reset_index(drop=True)

gdf = gdf.set_crs('EPSG:4326')
gdf.to_file(OUTPUT_CENSUS, driver='GeoJSON')
write_meta(OUTPUT_CENSUS, source_url=base_url, record_count=len(gdf),
           extra={'vintage': CENSUS_VINTAGE,
                  'temporal_coverage': f'{CENSUS_VINTAGE - 4}-{CENSUS_VINTAGE}'})

print(f"Saved {len(gdf)} tracts to {OUTPUT_CENSUS}")
print(f"Income range: ${gdf['median_income'].min():,.0f} - ${gdf['median_income'].max():,.0f}")

## 2. NCDOT Non-Motorist Crash Data

In [ ]:
from config import NCDOT_NONMOTORIST_SERVICE
from utils.freshness import write_meta

OUTPUT_CRASHES = RAW_DATA_DIR / 'ncdot_nonmotorist_durham.csv'

OUT_FIELDS = [
    'CrashID', 'CrashDate', 'CrashYear', 'CrashMonth', 'CrashHour',
    'Latitude', 'Longitude',
    'CrashSevr', 'CrashType', 'CrashTypGr', 'CrashAlcoh',
    'NM_Type', 'NM_Age', 'NM_Sex', 'NM_Race', 'NM_Inj', 'NM_AlcDrg',
    'NM_NumTot', 'NM_NumK', 'NM_NumA', 'NM_NumB', 'NM_NumC', 'NM_NumU',
    'DrvrAge', 'DrvrSex', 'DrvrRace', 'DrvrVehTyp',
    'SpeedLimit', 'RdClass', 'LightCond', 'Weather',
    'County', 'City',
]
MAX_RECORDS = 2000

base_params = {
    'where': "County='DURHAM'",
    'outFields': ','.join(OUT_FIELDS),
    'returnGeometry': 'false',
    'orderByFields': 'CrashID',
    'f': 'json',
}

print("Fetching NCDOT non-motorist crash data for Durham County...")
all_features = []
offset = 0
while True:
    params = {**base_params, 'resultOffset': offset}
    print(f"  Fetching records {offset}\u2013{offset + MAX_RECORDS}...")
    resp = requests.get(f"{NCDOT_NONMOTORIST_SERVICE}/query", params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if 'error' in data:
        raise RuntimeError(f"ArcGIS query error: {data['error']}")
    features = data.get('features', [])
    all_features.extend(f['attributes'] for f in features)
    if len(features) < MAX_RECORDS:
        break
    offset += MAX_RECORDS

if not all_features:
    raise RuntimeError("No records returned from NCDOT Feature Service")

crash_df = pd.DataFrame(all_features)

if 'CrashDate' in crash_df.columns:
    crash_df['CrashDate'] = pd.to_datetime(crash_df['CrashDate'], unit='ms').dt.strftime('%Y-%m-%d')

before = len(crash_df)
crash_df = crash_df.dropna(subset=['Latitude', 'Longitude'])
dropped = before - len(crash_df)
if dropped:
    print(f"  Dropped {dropped} records with missing coordinates")

crash_df.to_csv(OUTPUT_CRASHES, index=False)
year_min = int(crash_df['CrashYear'].min())
year_max = int(crash_df['CrashYear'].max())
write_meta(OUTPUT_CRASHES, source_url=NCDOT_NONMOTORIST_SERVICE,
           record_count=len(crash_df), extra={'year_range': [year_min, year_max]})

print(f"Saved {len(crash_df):,} crash records to {OUTPUT_CRASHES}")
print(f"  Years: {year_min}\u2013{year_max}")

## 3. OSM Infrastructure (osmnx)

In [ ]:
import json
import numpy as np
import osmnx as ox
from datetime import datetime, timezone
from config import DURHAM_BOUNDS, OVERPASS_API, OSM_INFRASTRUCTURE_FEATURES
from utils.freshness import write_meta

OUTPUT_OSM = RAW_DATA_DIR / 'osm_infrastructure.json'

# Configure osmnx — try the main endpoint first, fall back to kumi mirror
OVERPASS_MIRRORS = [
    OVERPASS_API,
    "https://overpass.kumi.systems/api/interpreter",
]

ox.settings.timeout = 180
ox.settings.log_console = False
ox.settings.max_query_area_size = 2_500_000_000  # ~2500 km²; keeps Durham to ~3 sub-queries

# Durham County bounding box — osmnx 2.x expects (west, south, east, north)
bbox = (
    DURHAM_BOUNDS['west'],
    DURHAM_BOUNDS['south'],
    DURHAM_BOUNDS['east'],
    DURHAM_BOUNDS['north'],
)

# All tags we care about in one query
tags = {
    'highway': ['crossing', 'traffic_signals', 'footway', 'cycleway', 'path'],
    'traffic_calming': True,
}

features_gdf = None
queried_at = datetime.now(timezone.utc).isoformat()

for mirror in OVERPASS_MIRRORS:
    try:
        print(f"Querying OSM via {mirror} ...")
        ox.settings.overpass_endpoint = mirror
        features_gdf = ox.features_from_bbox(bbox=bbox, tags=tags)
        print(f"  {len(features_gdf)} features returned")
        break
    except Exception as e:
        print(f"  Failed: {e}")
        features_gdf = None

if features_gdf is None:
    raise RuntimeError(
        "All Overpass mirrors failed. Try again later — "
        "check https://overpass-api.de/api/status for current load."
    )

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

def classify_row(row):
    highway = row.get('highway', '')
    if pd.isna(highway):
        highway = ''
    if isinstance(highway, list):
        highway = highway[0] if highway else ''
    if highway == 'crossing':        return 'crossings'
    if highway == 'traffic_signals': return 'traffic_signals'
    if highway == 'footway':         return 'footways'
    if highway in ('cycleway', 'path') and row.get('bicycle') != 'no':
        return 'bike_infra'
    if pd.notna(row.get('traffic_calming')): return 'speed_calming'
    return None

# Convert to point geometries (use centroid for polygons/lines)
records = []
for _, row in features_gdf.iterrows():
    category = classify_row(row)
    if category is None:
        continue
    geom = row.geometry
    if geom is None:
        continue
    point = geom if geom.geom_type == 'Point' else geom.centroid
    records.append({'category': category, 'geometry': point})

if not records:
    raise RuntimeError("No infrastructure elements classified from OSM response")

infra_gdf = gpd.GeoDataFrame(records, crs='EPSG:4326')
print(f"Classified {len(infra_gdf)} elements:")
for cat, count in infra_gdf['category'].value_counts().items():
    print(f"  {cat}: {count}")

# Spatial-join to census tracts
tracts_gdf = gpd.read_file(OUTPUT_CENSUS)
if tracts_gdf.crs is None:
    tracts_gdf = tracts_gdf.set_crs('EPSG:4326')
tracts_projected = tracts_gdf.to_crs(epsg=3857)
tracts_gdf['area_km2'] = tracts_projected.geometry.area / 1e6

joined = gpd.sjoin(infra_gdf, tracts_gdf[['tract_id', 'geometry']], how='left', predicate='within')

categories = list(OSM_INFRASTRUCTURE_FEATURES.keys())
tract_ids = tracts_gdf['tract_id'].unique()
tract_data = {tid: {cat: 0 for cat in categories} for tid in tract_ids}
for _, row in joined.iterrows():
    tid = row.get('tract_id')
    if pd.isna(tid):
        continue
    tract_data[tid][row['category']] += 1

In [ ]:
# Compute per-tract density scores and save
# Density is per 1,000 residents, not per km².
# Per-area density is highest in urban cores (which are also lower-income in Durham),
# producing an inverted income gradient. Per-capita removes that confound.
rows = []
for tid in tract_ids:
    area_km2 = tracts_gdf.loc[tracts_gdf['tract_id'] == tid, 'area_km2'].iloc[0]
    population = tracts_gdf.loc[tracts_gdf['tract_id'] == tid, 'total_population'].iloc[0]
    row = {'tract_id': tid, 'area_km2': float(area_km2)}
    for cat in categories:
        count = tract_data[tid][cat]
        row[f'{cat}_count'] = count
        # per 1,000 residents
        row[f'{cat}_density'] = (count / population * 1000) if population > 0 else 0.0
    rows.append(row)

result_df = pd.DataFrame(rows)

for cat in categories:
    density_col = f'{cat}_density'
    # log1p before normalizing to compress outlier effect from dense downtown tracts
    log_col = np.log1p(result_df[density_col])
    col_min = log_col.min()
    col_range = log_col.max() - col_min
    if col_range > 0:
        result_df[f'{cat}_norm'] = (log_col - col_min) / col_range
    else:
        result_df[f'{cat}_norm'] = 0.0

result_df['osm_infrastructure_score'] = sum(
    result_df[f'{cat}_norm'] * OSM_INFRASTRUCTURE_FEATURES[cat]['weight']
    for cat in categories
)
result_df['osm_infrastructure_score'] = np.clip(
    result_df['osm_infrastructure_score'], 0.05, 0.95
)

output = {
    '_provenance': {
        'data_type': 'real',
        'source': 'OpenStreetMap via osmnx',
        'queried_at': queried_at,
        'features_queried': list(OSM_INFRASTRUCTURE_FEATURES.keys()),
        'bounds': DURHAM_BOUNDS,
        'total_elements': len(infra_gdf),
    },
    'totals': {cat: int(result_df[f'{cat}_count'].sum()) for cat in categories},
    'tracts': result_df.to_dict(orient='records'),
}

with open(OUTPUT_OSM, 'w') as f:
    json.dump(output, f, indent=2)

write_meta(OUTPUT_OSM, source_url=ox.settings.overpass_endpoint,
           record_count=len(infra_gdf), extra={'queried_at': queried_at})

print(f"Saved OSM infrastructure to {OUTPUT_OSM}")
print(f"  Mean infrastructure score: {result_df['osm_infrastructure_score'].mean():.3f}")

## 4. Publish Artifacts

In [ ]:
colab_utils.publish_artifacts(
    paths=[
        "backend/data/raw/durham_census_tracts.geojson",
        "backend/data/raw/ncdot_nonmotorist_durham.csv",
        "backend/data/raw/osm_infrastructure.json",
    ],
    message="data: refresh raw Durham data",
    repo_dir=repo,
)